In [2]:

import time, numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))
import sys, sklearn; print("Python", sys.version.split()[0], "| sklearn", sklearn.__version__)

df = pd.read_csv('/content/drive/MyDrive/project/data.csv')
df = df.dropna(subset=['review','sentiment']).copy()
df['review'] = df['review'].astype(str)
df['sentiment'] = df['sentiment'].str.lower().str.strip()

print("\n=== DATA ===")
print(df['sentiment'].value_counts())
print("Duplicate reviews:", df['review'].duplicated().sum())
print(df['review'].str.split().str.len().describe())

X = df['review']
enc = LabelEncoder(); y = enc.fit_transform(df['sentiment'])
print("Classes:", enc.classes_)
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

def build(n_features):
    return Sequential([
        Input(shape=(n_features,)),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(3, activation='softmax')])

def run(name, vec, epochs=5, early_stop=False, seed=42):
    tf.keras.utils.set_random_seed(seed)
    A = vec.fit_transform(x_train); B = vec.transform(x_test)
    model = build(A.shape[1])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    cbs = [EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)] if early_stop else []
    t0 = time.time()
    h = model.fit(A.toarray(), y_train, epochs=epochs, batch_size=64,
                  validation_split=0.2, callbacks=cbs, verbose=0)
    total = time.time() - t0
    proba = model.predict(B.toarray(), verbose=0); pred = proba.argmax(1)
    print(f"\n=================== {name} ===================")
    print("Features:", A.shape[1], "| train shape:", A.shape, "| test shape:", B.shape)
    print("Trainable params:", model.count_params())
    print("Epochs run:", len(h.history['loss']), "| total train time (s):", round(total, 1))
    for k, v in h.history.items(): print(k, [round(float(x), 4) for x in v])
    print("TEST ACCURACY:", round(accuracy_score(y_test, pred), 4))
    print(classification_report(y_test, pred, target_names=enc.classes_, digits=4))
    print("Confusion matrix (rows=true, cols=pred; order:", list(enc.classes_), ")")
    print(confusion_matrix(y_test, pred))
    return model, vec, pred

m1, v1, p1 = run("A) Baseline FNN (5000 unigrams)", TfidfVectorizer(max_features=5000))
m2, v2, p2 = run("B) FNN-Bigram (uni+bigrams, 10000, min_df=2, sublinear)",
                 TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True, min_df=2))
m3, v3, p3 = run("C) FNN-Bigram + early stopping (max 20 epochs)",
                 TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True, min_df=2),
                 epochs=20, early_stop=True)

print("\n=== Test reviews where baseline and FNN-Bigram DISAGREE (first 8) ===")
idx = np.where(p1 != p2)[0]; print("Number of disagreements:", len(idx))
for i in idx[:8]:
    print(f"- {x_test.iloc[i][:150]!r}\n  true={enc.classes_[y_test[i]]} | baseline={enc.classes_[p1[i]]} | bigram={enc.classes_[p2[i]]}")

print("\n=== Sample predictions (FNN-Bigram) ===")
samples = ["The product is amazing and excellent", "Very bad experience and poor quality",
           "The product is okay, nothing special", "The film was not good", "Not bad at all"]
pr = m2.predict(v2.transform(samples).toarray(), verbose=0)
for s_, p_ in zip(samples, pr): print(s_, "->", enc.classes_[p_.argmax()], np.round(p_, 3))

TF 2.21.0 | GPU: []
Python 3.13.18 | sklearn 1.6.1

=== DATA ===
sentiment
positive    3750
negative    3311
neutral     3084
Name: count, dtype: int64
Duplicate reviews: 2562
count    10145.000000
mean        14.361755
std          6.287352
min          1.000000
25%         10.000000
50%         14.000000
75%         18.000000
max         50.000000
Name: review, dtype: float64
Classes: ['negative' 'neutral' 'positive']

=================== A) Baseline FNN (5000 unigrams) ===================
Features: 5000 | train shape: (8116, 5000) | test shape: (2029, 5000)
Trainable params: 648579
Epochs run: 5 | total train time (s): 5.6
accuracy [0.7763, 0.9455, 0.9858, 0.9949, 0.9988]
loss [0.7227, 0.1649, 0.0564, 0.0247, 0.0118]
val_accuracy [0.8996, 0.9156, 0.9156, 0.9144, 0.9046]
val_loss [0.3009, 0.2282, 0.2396, 0.2598, 0.2895]
TEST ACCURACY: 0.9202
              precision    recall  f1-score   support

    negative     0.9431    0.9260    0.9345       662
     neutral     0.9261    0.8736  

In [3]:
res = {"baseline": [], "bigram": []}
for seed in [1, 2, 3, 4, 5]:
    for name, vec in [("baseline", TfidfVectorizer(max_features=5000)),
                      ("bigram", TfidfVectorizer(max_features=10000, ngram_range=(1,2), sublinear_tf=True, min_df=2))]:
        tf.keras.utils.set_random_seed(seed)
        A = vec.fit_transform(x_train); B = vec.transform(x_test)
        mdl = build(A.shape[1])
        mdl.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        mdl.fit(A.toarray(), y_train, epochs=5, batch_size=64, validation_split=0.2, verbose=0)
        res[name].append(accuracy_score(y_test, mdl.predict(B.toarray(), verbose=0).argmax(1)))
for k, v in res.items():
    print(k, [round(x, 4) for x in v], "mean=%.4f std=%.4f" % (np.mean(v), np.std(v)))
from scipy.stats import ttest_rel
print("Paired t-test p-value:", ttest_rel(res["bigram"], res["baseline"]).pvalue)

baseline [0.9216, 0.9207, 0.9182, 0.9266, 0.9216] mean=0.9217 std=0.0027
bigram [0.9305, 0.9335, 0.93, 0.9315, 0.9295] mean=0.9310 std=0.0014
Paired t-test p-value: 0.002810077136654889
